# CTRLxAI December 2026 IAB: Learning PID gains

## Introduction
In this workshop, we will dive into one of the CTRLxAI algorithms for distilling a reinforcement learning agent into Multi-Input-Multi-Output (MIMO) PIDs. One the algorithms presented in the morning, Local PID Network (LoPIDN), will be used to find controllers for two simple benchmarks: Pendulum and Mountaincar. We briefly repeat the basics behind the algorithm after which you will be able to execute and tweak the hyper parameters. Do not hesitate to ask questions during the
session.


## A brief recap

Local PID Network (LoPIDN) is an algorithm that learns to mimic a black box RL agent using a set of MIMO PIDs. Inspired by Local Model Networks (LMN), we use the critic value network of the RL agent as the validity function, determining how much each controller contributes to the total control signal.

The gains of each controller are learned from a dataset of state-action pairs, generated by the RL agent. Using gradient descent, a loss metric (the mean-squared error between RL and controller output) is minimized while the loss contribution of each controller is weighted by the critic in the backward pass.

The result is a set of PID's that, combined with the critic, van be used to control the system in a similar manner to the original RL agent.


## Methodology
During this workshop, we will perform the following steps:

1. Inspect the performance of a pre-trained RL agent

2. Generate a dataset from the RL agent

3. Train a CriticLMN on the dataset

4. Validate and inspect the found PIDs

We will try to first learn with a single PID, observe the results and increase the amount of controllers. Aside this, we will vary a few hyperparameters that could improve on the training process.


## Setup
The following cells pull the project into the colab and set up the virtual environment.

In [ ]:
%cd /content
!rm -r sample_data
!git clone https://github.com/SenneDeproost/IAB_DEC26.git

In [ ]:
%cd IAB_DEC26/
!uv sync
!uv pip uninstall --system gym
!pip install stable_baselines3 gymnasium

In [ ]:
import sys
sys.path.append('./src/lopidn')
import gymnasium as gym
import random
import torch as th
from tqdm import tqdm
import yaml
import numpy as np

import imageio
from IPython.display import Image, display

import stable_baselines3 as sb3
from stable_baselines3 import TD3
from stable_baselines3.common.evaluation import evaluate_policy

from config import RunConfig
from data import generate_dataset
from optimization import optimize_controller, optimize_LMN
import utils
from models import LocalModelNetwork, CriticLMN

# Environment 1: Pendulum

We first start with the Pendulum environment, a classic control problem where the goal is to swing a pendulum up so that it stays upright, pointing vertically upwards, with the least amount of control effort.

### *Observation Space*

The observation is a 3-dimensional vector representing the state of the pendulum:

| Index | Observation | Min | Max | Description |
| :---: | :--- | :---: | :---: | :--- |
| 0 | `cos(theta)` | -1.0 | 1.0 | Cosine of the pendulum's angle |
| 1 | `sin(theta)` | -1.0 | 1.0 | Sine of the pendulum's angle |
| 2 | `theta_dot` | -8.0 | 8.0 | Angular velocity of the pendulum |

*Note: `theta` is the angle where 0 (or $2\pi$) represents the upright position, pointing straight up.*

### *Action Space*

The action space is a 1-dimensional continuous space representing the torque applied to the joint:

| Index | Action | Min | Max | Description |
| :---: | :--- | :---: | :---: | :--- |
| 0 | Torque | -2.0 | 2.0 | Joint effort applied to swing the pendulum |

### *Reward Function*

The reward function is continuous and penalizes the pendulum for being far from the upright position, rotating too fast, or using too much effort (torque). It is calculated at each step as:

$$r = - (\theta^2 + 0.1 \cdot \dot{\theta}^2 + 0.001 \cdot a^2)$$

Where:
* $\theta$ is the normalized angle of the pendulum (mapped to the range $[-\pi, \pi]$) where $0$ is upright.
* $\dot{\theta}$ is the angular velocity (`theta_dot`).
* $a$ is the applied torque (action).

The maximum possible reward per step is $0.0$ (when the pendulum is perfectly upright, stationary, and no torque is being applied).

In [ ]:
# Load config file
with open(f'./config/Pendulum.yaml', "r") as f:
    args = yaml.safe_load(f)
args = RunConfig(**args)
args.load_modules()

# Create environment
env = gym.make(args.env_id)

## Inspect

We first inspect the performance of the RL agent, both in terms of mean return and visual.

### *Performance*

In [ ]:
# 1. Load the pre-trained TD3 agent
model_path = f'./{args.rl_agent}'
rl_agent = TD3.load(model_path, env=env)

# 2. Validate using mean return and standard deviation
mean_reward, std_reward = evaluate_policy(rl_agent, env, n_eval_episodes=args.validation_n_episodes, warn=False)
print(f'RL Agent Mean Return: {mean_reward} ± {std_reward}')

### *Visual*

In [ ]:
utils.create_rl_gif(rl_agent, args, './pendulum_rl.gif', seed=0)

In [ ]:
# Create dataset
dataset = generate_dataset(env_id=args.env_id,
                           agent=f'./{args.rl_agent}',
                           save_path=f'./trained_rl/datasets/{args.algorithm}/{args.env_id}.pkl',
                           algorithm=args.algorithm_class,
                           n_episodes=args.data_n_episodes)

## Optimize 1 PID
We first optimize a single PID using the technique.

In [ ]:
args.n_controllers = 1

In [ ]:
# Initialize router
router = CriticLMN(n_inputs=env.observation_space.shape[0],
                   n_outputs=env.action_space.shape[0],
                   n_controllers=args.n_controllers,
                   controller_class=args.controller_class,
                   controller_target=args.target,
                   controller_bound = (env.action_space.low, env.action_space.high),
                   critic=dataset.critic)

In [ ]:
# Call optimization function
optimize_LMN(model=router,
             data=dataset,
             args=args)

In [ ]:
# Run several validation runs in environment
val_ret = utils.validate(controller=router,
                         env_id=args.env_id,
                         render_env=False,
                         n_episodes=args.validation_n_episodes)
print(f'Mean return: {np.mean(val_ret)} ± {np.std(val_ret)}')

In [ ]:
utils.create_pid_gif(router, args, './pendulum_1_pid.gif', seed=0)

In [ ]:
router.get_gains()

### Optimize 2 PID

In [ ]:
args.n_controllers = 2
args.learning_rate = 0.05
args.n_epochs_training = 30

In [ ]:
# Initialize router
router = CriticLMN(n_inputs=env.observation_space.shape[0],
                   n_outputs=env.action_space.shape[0],
                   n_controllers=args.n_controllers,
                   controller_class=args.controller_class,
                   controller_target=args.target,
                   controller_bound = (env.action_space.low, env.action_space.high),
                   critic=dataset.critic)

In [ ]:
# Call optimization function
optimize_LMN(model=router,
             data=dataset,
             args=args)

In [ ]:
# Run several validation runs in environment
val_ret = utils.validate(controller=router,
                         env_id=args.env_id,
                         render_env=False,
                         n_episodes=args.validation_n_episodes)
print(f'Mean return: {np.mean(val_ret)} ± {np.std(val_ret)}')

In [ ]:
utils.plot_routing(router, args.env_id, seed=0)

In [ ]:
utils.create_pid_gif(router, args, './pendulum_2_pid.gif', seed=0)

# Environment 2: Mountain Car

The Mountain Car Continuous environment is a classic control challenge where a underpowered car must drive up a steep hill. Because the engine is not strong enough to climb the slope directly, the car must build potential energy by driving back and forth up the opposite hill.

## Observation Space

The observation is a 2-dimensional vector representing the state of the car:

| Index | Observation | Min | Max | Description |
| :---: | :--- | :---: | :---: | :--- |
| 0 | Position | -1.2 | 0.6 | Position of the car along the 1D track |
| 1 | Velocity | -0.07 | 0.07 | Velocity of the car |

*Note: The goal position is at `0.45` on the right hill.*

## Action Space

The action is a 1-dimensional continuous value representing the directional force applied to the car:

| Index | Action | Min | Max | Description |
| :---: | :--- | :---: | :---: | :--- |
| 0 | Force | -1.0 | 1.0 | Continuous motor force applied to the car (negative is left, positive is right) |

## Reward Function

The reward at each step is designed to encourage reaching the goal quickly while minimizing energy/force expenditure:

* Reaching the goal (position $\ge 0.45$) yields a one-time reward of **$+100$**.
* At each step, a penalty of **$-0.1 \cdot a^2$** is applied, where $a$ is the action force applied.

The goal is to reach the target position with minimal fuel use and as fast as possible.

In [ ]:
# Load config file
with open(f'./config/MountainCarContinuous.yaml', "r") as f:
    args = yaml.safe_load(f)
args = RunConfig(**args)
args.load_modules()

# Create environment
env = gym.make(args.env_id)

## Inspect

We first inspect the performance of the RL agent, both in terms of mean return and visual.

### *Performance*

In [ ]:
# 1. Load the pre-trained TD3 agent
model_path = f'./{args.rl_agent}'
rl_agent = TD3.load(model_path, env=env)

# 2. Validate using mean return and standard deviation
mean_reward, std_reward = evaluate_policy(rl_agent, env, n_eval_episodes=args.validation_n_episodes, warn=False)
print(f'RL Agent Mean Return: {mean_reward} ± {std_reward}')

### *Visual*

In [ ]:
utils.create_rl_gif(rl_agent, args, './mountaincar_rl.gif', seed=0)

In [ ]:
# Create dataset
dataset = generate_dataset(env_id=args.env_id,
                           agent=f'./{args.rl_agent}',
                           save_path=f'./trained_rl/datasets/{args.algorithm}/{args.env_id}.pkl',
                           algorithm=args.algorithm_class,
                           n_episodes=args.data_n_episodes)

## Optimize 1 PID
We first optimize a single PID using the technique.

In [ ]:
args.n_controllers = 1

In [ ]:
# Initialize router
router = CriticLMN(n_inputs=env.observation_space.shape[0],
                   n_outputs=env.action_space.shape[0],
                   n_controllers=args.n_controllers,
                   controller_class=args.controller_class,
                   controller_target=args.target,
                   controller_bound = (env.action_space.low, env.action_space.high),
                   critic=dataset.critic)

In [ ]:
# Call optimization function
optimize_LMN(model=router,
             data=dataset,
             args=args)

In [ ]:
# Run several validation runs in environment
val_ret = utils.validate(controller=router,
                         env_id=args.env_id,
                         render_env=False,
                         n_episodes=args.validation_n_episodes)
print(f'Mean return: {np.mean(val_ret)} ± {np.std(val_ret)}')

In [ ]:
utils.create_pid_gif(router, args, './mountaincar_1_pid.gif', seed=0)

### Optimize 2 PID

In [ ]:
args.n_controllers = 2

In [ ]:
# Initialize router
router = CriticLMN(n_inputs=env.observation_space.shape[0],
                   n_outputs=env.action_space.shape[0],
                   n_controllers=args.n_controllers,
                   controller_class=args.controller_class,
                   controller_target=args.target,
                   controller_bound = (env.action_space.low, env.action_space.high),
                   critic=dataset.critic)

In [ ]:
# Call optimization function
optimize_LMN(model=router,
             data=dataset,
             args=args)

In [ ]:
# Run several validation runs in environment
val_ret = utils.validate(controller=router,
                         env_id=args.env_id,
                         render_env=False,
                         n_episodes=args.validation_n_episodes)
print(f'Mean return: {np.mean(val_ret)} ± {np.std(val_ret)}')

In [ ]:
utils.plot_routing(router, args.env_id, seed=0)

In [ ]:
utils.create_pid_gif(router, args, './mountaincar_2_pid.gif', seed=0)